# MV variant comparison

This notebook compares the MV variants of one LV version with a reference variant, and the same MV variant on two
LV versions with different station sets (greenfield stations of the LV stage against brownfield OSM/LoD2
stations).

An MV variant is defined by its parameters, stored in `mv_variant.mv_parameters`: the `MV_*` keys
(`pylovo-mv generate --set`) and the constants of the stage (`MvParameters.replace` in Python). The notebook finds
out in which parameters each variant differs from the reference and groups the comparisons:

- **solver**: the local search after the savings construction (`local_search_rounds`, `neighbours`,
  `ls_time_guard_s`);
- **routing**: the constants of the routing graph and the street routes (`routing.*`, e.g. `osm_line_cost_factor`
  of the OSM MV lines or `busbar_exit_radius_m`);
- **sources**: the constants of the HV/MV substation sites and their supply areas (`sources.*`);
- **stubs and links**: the rural stubs, the normally-open links inside a district and the ties between districts
  (`stub_*`, `link_*`, `tie_*`);
- **equipment and rules**: everything else (conductor types, planning utilization, nominal voltage, ring size).

Districts of two variants are paired by their substation (nearest substation within 200 m), rings are compared by
their distributions (Wasserstein distance) and by the adjusted Rand index of "same ring" membership of the
stations.

**Prerequisites**

- pylovo with the `notebooks` extra (`uv sync --extra notebooks`) and this notebook running on the kernel of the
  project environment (`.venv`), see `notebook_tutorials/README.md`.
- A `.env` in the repository root that points to the database with the MV results, and MV variants of the LV
  version (`pylovo-mv list --version <version>`; `0_mv_generation.ipynb` shows how to generate them).
- The notebook only reads from the database: its client runs read-only transactions.

In [ ]:
# Parameters
version_id = "mv2"  # LV version of the MV test database; None: VERSION_ID of config/config_generation_lv.yaml
variants = None          # MV variants to compare; None: all done variants of the version that are not retired
reference = None         # None: the default variant (MV_VARIANT_ID if the version has it, else the newest done
                         # variant that is not retired) if selected, else the first; an explicit id must be selected
compare_versions = None  # two LV versions with different station sets, e.g. ["v_green", "v_brown"]; None: skip
compare_variant = None   # MV variant of the station-set comparison; None: the reference variant
regions = None           # region parts to keep (1 = largest), e.g. [1]; None: all
plz = None               # keep only the districts with nodes in these PLZ; None: all
grid_id = None           # district of the side-by-side maps (id in the reference); None: the one with most stations
basemap = False          # OpenStreetMap tiles under the maps (needs internet access)
dbname = "infdb_mv"  # expected database of the .env (the MV test database); None: no check
repo_root = None         # repository root; None: two levels above this notebook

## Setup

The first cell changes to the repository root, so that pylovo reads the `.env` and `config/` of this checkout
(and writes its log to `log/`), and checks the database name when `dbname` is set. `mv_notebook_utils.py` next to
this notebook holds the shared helpers of the MV notebooks.

In [ ]:
import os
import sys
from pathlib import Path

if repo_root is None:
    here = Path.cwd().resolve()
    repo_root = next((p for p in (here, *here.parents)
                      if (p / "pyproject.toml").is_file() and (p / "src" / "pylovo").is_dir()), None)
    if repo_root is None:
        raise RuntimeError("Set repo_root to the MV checkout before running this notebook.")
repo_root = Path(repo_root).expanduser().resolve()

# A kernel registered for another checkout can silently import its pylovo installation.
import importlib.util

spec = importlib.util.find_spec("pylovo")
expected = repo_root / "src" / "pylovo" / "__init__.py"
if spec is None or spec.origin is None or Path(spec.origin).resolve() != expected:
    raise RuntimeError(
        f"Select the notebook kernel at {repo_root / '.venv' / 'bin' / 'python'}, "
        "restart it, then Run All. The current kernel imports pylovo from "
        f"{None if spec is None else spec.origin!r}."
    )
os.chdir(repo_root)
sys.path.insert(0, str(repo_root / "notebook_tutorials" / "mv_grids"))

from pylovo import config_loader

if dbname is not None:
    assert config_loader.DBNAME == dbname, f"the .env points to {config_loader.DBNAME!r}, not {dbname!r}"

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import mv_notebook_utils as mvu
from pylovo.analysis import mv_validation
from pylovo.plotting import mv_grid as mvplot
from pylovo.plotting.utils import get_color_map

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 200)

dbc = mvu.read_only_client()
version_id, variants, listing = mvu.resolve(dbc, version_id, variants, select="current")
print(f"Database {config_loader.DBNAME}, LV version {version_id}, MV variants: {', '.join(variants) or 'none'}")

In [ ]:
assert variants, "no MV variants to analyse"
plz_union = sorted({int(p) for p in listing.loc[listing["mv_variant_id"].isin(variants), "plz_list"].explode()})
parts = mvu.region_parts(dbc, plz_union)
data = {v: mvu.load_variant(dbc, version_id, v, parts=parts, regions=regions, plz=plz) for v in variants}
colors = get_color_map(variants)
districts = pd.concat([mvu.district_table(v) for v in data.values()], ignore_index=True)
rings = pd.concat([mvu.ring_table(v) for v in data.values()], ignore_index=True)
print(f"{len(parts)} region parts; districts per variant: "
      + ", ".join(f"{k} {len(v.grids)}" for k, v in data.items()))

if reference is not None and reference not in data:
    raise ValueError(f"reference {reference!r} is not among the selected variants")
default = mvu.default_variant(listing)
ref = reference or (default if default in data else variants[0])
anchors = pd.concat([mvu.demand_anchors(dbc, v) for v in data.values()], ignore_index=True)
clipped = mvu.clipped_table(list(data.values()), by_region=False)
totals = mvu.summary_table(districts, rings, by=["variant"])
totals = totals.join(anchors[anchors["scope"] == "all"].set_index("variant")[["kw_per_inhabitant", "population"]])
totals = totals.join(clipped.xs(0, level="region")[["km_per_km2"]])
print(f"Reference variant: {ref}")
print("Inventory units: circuit km counts installed parallel circuits; route km counts each routed line once.")

## Parameters of the variants

The parameters in which the variants differ, and for every variant the parameters that differ from the
reference with their comparison dimension.

In [ ]:
display(mvu.parameter_table(list(data.values())))
flat = {name: mvu.variant_parameters(v.params) for name, v in data.items()}
rows = []
for name in variants:
    if name == ref:
        continue
    keys = sorted(k for k in set(flat[name]) | set(flat[ref]) if str(flat[name].get(k)) != str(flat[ref].get(k)))
    rows.append({"variant": name, "dimensions": sorted({mvu.dimension_of(k) for k in keys}),
                 "changed parameters": ", ".join(f"{k} = {flat[name].get(k)}" for k in keys)})
differences = pd.DataFrame(rows, columns=["variant", "dimensions", "changed parameters"]).set_index("variant")
groups = {dim: [ref] + [n for n, dims in differences["dimensions"].items() if dim in dims]
          for dim, _ in mvu.DIMENSIONS}
display(differences.assign(dimensions=differences["dimensions"].map(", ".join)))
print("Comparison groups: " + "; ".join(f"{dim}: {', '.join(names)}" for dim, names in groups.items()))

## Side-by-side metrics

The totals of every variant (as in `1_mv_results_overview.ipynb`), then the change against the reference in
percent, and small multiples of the main metrics.

In [ ]:
mvu.show_blocks(totals)

In [ ]:
key = ["route_km", "circuit_km", "cable_km", "overhead_km", "trench_km", "km_per_station", "km_per_supply_point", "km_per_km2",
       "n_rings", "ring_members_mean", "half_ring_km_mean", "ring_km_mean", "n_stub_nodes", "n_feeder_bays",
       "max_loading_percent", "max_loading_n1_percent", "max_drop_percent", "max_drop_n1_percent",
       "n_repairs", "n_band_conflicts", "runtime_s"]
change = (totals[key] / totals.loc[ref, key] - 1.0) * 100.0
display(Markdown(f"**Change against {ref} (%)**"))
display(change.T.round(1).replace(-0.0, 0.0))
fig = mvplot.plot_variant_summary(totals.reset_index(), metrics=[
    "circuit_km", "cable_share", "km_per_station", "ring_members_mean", "half_ring_km_mean", "n_stub_nodes",
    "max_loading_n1_percent", "max_drop_n1_percent", "n_band_conflicts"], ncols=3)

## Distributions

Ring members, half-ring and ring lengths of the open rings and the circuit km per supply point of the districts,
per variant with the literature ranges, and the Wasserstein distance of each distribution to the reference
(`wasserstein_rel`: divided by the reference mean).

In [ ]:
ring_rows = rings[rings["kind"] == "ring"].rename(columns={"n_members": "ring_members", "length_km": "ring_km"})
halves = pd.concat([ring_rows[["variant", "area_class", c]].rename(columns={c: "feeder_km"})
                    for c in ("half_a_km", "half_b_km")], ignore_index=True)
sources = {"ring_members": ring_rows, "feeder_km": halves, "ring_km": ring_rows, "km_per_supply_point": districts}
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, (metric, frame) in zip(axes.flat, sources.items()):
    mvplot.plot_metric_distribution(frame, metric, by="variant", order=variants, ax=ax,
                                    title=f"{mvplot.metric_label(metric)}")
fig.tight_layout()
tables = []
for metric, frame in sources.items():
    w = mv_validation.wasserstein_table({n: frame.loc[frame["variant"] == n, metric] for n in variants}, ref)
    tables.append(w.set_index("variant")[["mean", "median", "p90", "wasserstein", "wasserstein_rel"]]
                  .add_prefix(f"{metric}: "))
display(pd.concat(tables, axis=1).T.round(3))

## Paired districts and ring membership

Districts with the same substation in a variant and in the reference: the median and the range of the paired
differences (variant minus reference). The adjusted Rand index (ARI) compares which stations share a ring
(stations keyed by their LV grid): 1 means the same rings, 0 a random agreement.

In [ ]:
paired_metrics = ["circuit_km", "km_per_station", "ring_members_mean", "half_ring_km_mean", "cable_share",
                  "n_stub_nodes", "max_drop_n1_percent", "max_loading_n1_percent", "n_band_conflicts"]
base = districts[districts["variant"] == ref].set_index("mv_grid_id")
paired = []
for name in variants:
    if name == ref:
        continue
    other = districts[districts["variant"] == name].set_index("mv_grid_id")
    for row in mvu.district_pairs(data[ref], data[name]).itertuples():
        paired.append({"variant": name, "mv_grid_id_ref": row.mv_grid_id_ref, "mv_grid_id": row.mv_grid_id,
                       **{m: other.at[row.mv_grid_id, m] - base.at[row.mv_grid_id_ref, m] for m in paired_metrics},
                       "circuit_km_change_percent": 100 * (other.at[row.mv_grid_id, "circuit_km"]
                                                           / base.at[row.mv_grid_id_ref, "circuit_km"] - 1)})
paired = pd.DataFrame(paired)
if paired.empty:
    print("No other variant to pair with the reference")
else:
    summary = paired.groupby("variant").agg(
        n_pairs=("mv_grid_id", "size"),
        **{f"{m} median": (m, "median") for m in paired_metrics + ["circuit_km_change_percent"]})
    display(mvu.pretty(summary.T, rows=True))
    fig = mvplot.plot_metric_distribution(paired, "circuit_km_change_percent", by="variant",
                                          order=[v for v in variants if v != ref], figsize=(10, 5),
                                          title=f"Circuit km per district against {ref} (%)")


def ring_labels(v):
    st = v.nodes[(v.nodes["kind"] == "station") & v.nodes["ring_id"].notna() & v.nodes["grid_result_id"].notna()]
    return {int(g): (int(m), int(r)) for g, m, r in zip(st["grid_result_id"], st["mv_grid_id"], st["ring_id"])}


labels_ref = ring_labels(data[ref])
ari = pd.DataFrame([{"variant": n, "n_common_stations": len(set(labels_ref) & set(ring_labels(v))),
                     "ari_same_ring": mv_validation.ring_membership_ari(labels_ref, ring_labels(v))}
                    for n, v in data.items()]).set_index("variant")
display(ari.round(3))

## Solver

Total ring length after the savings construction and after the local search per district, the local-search gain
and the solver runtime (`mv_grid.metrics`), for the reference and the variants that differ from it in a constant of
the local search.

In [ ]:
names = groups["solver"]
if len(names) == 1:
    print(f"No variant differs from {ref} in the local search; showing the reference.")
d = districts[districts["variant"].isin(names)]
table = d.groupby("variant", sort=False).agg(
    km_after_savings=("length_savings_m", lambda s: s.sum() / 1000), km_final=("length_final_m", lambda s: s.sum() / 1000),
    solver_runtime_s=("solver_runtime_s", "sum"), circuit_km=("circuit_km", "sum"),
    ring_members_mean=("ring_members_mean", "mean"), max_drop_n1_percent=("max_drop_n1_percent", "max"))
table["local_search_gain_percent"] = 100 * (1 - table["km_final"] / table["km_after_savings"])
display(mvu.pretty(table))
fig = mvplot.plot_solver_convergence(d[["variant", "length_savings_m", "length_final_m", "solver_runtime_s"]])

## Routing

The street routes against the straight lines between their ends, for the reference and the variants that differ
from it in a routing constant. Route length counts each line once and defines the detour factor; circuit
inventory counts parallel conductors. Trench length is the union of street pieces used by cables and is distinct
from both.

In [ ]:
names = groups["routing"]
if len(names) == 1:
    print(f"No variant differs from {ref} in routing; showing the reference.")
rows = []
for name in names:
    v = data[name]
    lines = v.lines[v.lines["role"] != "tie"]
    parallel = pd.to_numeric(lines["parallel"], errors="coerce").fillna(1)
    route_km = lines["length_km"].sum()
    circuit_km = (lines["length_km"] * parallel).sum()
    cable_km = (lines["cable_km"] * parallel).sum()
    straight_km = lines["straight_km"].sum()
    rows.append({"variant": name, "osm_line_cost_factor": v.snapshot("routing.osm_line_cost_factor"),
                 "route_km": route_km, "circuit_km": circuit_km, "straight_km": straight_km,
                 "detour_factor": route_km / straight_km if straight_km else np.nan,
                 "cable_km": cable_km, "trench_km": v.grids["trench_km"].sum(),
                 "trench_per_cable_km": v.grids["trench_km"].sum() / cable_km if cable_km else np.nan,
                 "km_per_station": circuit_km / v.grids["n_stations"].sum(),
                 "cable_share": cable_km / circuit_km if circuit_km else np.nan})
display(mvu.pretty(pd.DataFrame(rows).set_index("variant")))


## Sources: substations and supply areas

For the reference and the variants that differ from it in a constant of the substation sites or supply areas:
substations used, their eGon-data match, district areas, supply points and peaks per substation, and the
intersection over union (IoU) of the paired district cells with the reference.

In [ ]:
names = groups["sources"]
if len(names) == 1:
    print(f"No variant differs from {ref} in the sources; showing the reference.")
rows = []
for name in names:
    v = data[name]
    d = districts[districts["variant"] == name]
    cells = v.grids.set_index("mv_grid_id").geometry
    ref_cells = data[ref].grids.set_index("mv_grid_id").geometry
    pairs = mvu.district_pairs(data[ref], v)
    ious = [ref_cells[a].intersection(cells[b]).area / ref_cells[a].union(cells[b]).area
            for a, b in zip(pairs["mv_grid_id_ref"], pairs["mv_grid_id"])
            if ref_cells.get(a) is not None and cells.get(b) is not None]
    rows.append({"variant": name, "n_districts": len(d),
                 "n_substations_in_region": int(d["substation_in_region"].sum()),
                 "n_egon_match": int(v.substations.loc[v.substations["used"], "egon_bus_id"].notna().sum()),
                 "district_km2_median": d["area_km2"].median(), "district_km2_max": d["area_km2"].max(),
                 "supply_points_per_substation": d["n_supply_points"].mean(),
                 "peak_mw_mean": d["peak_load_mw"].mean(), "peak_mw_max": d["peak_load_mw"].max(),
                 "hv_mv_unit_mva_mean": d["hv_mv_unit_mva"].mean(),
                 "n_paired_with_reference": len(pairs), "cell_iou_mean": np.mean(ious) if ious else np.nan})
display(mvu.pretty(pd.DataFrame(rows).set_index("variant")))

## Stubs, links and ties

For the reference and the variants that differ from it in a constant of the stubs, links or ties: cable share,
overhead length, stub nodes and stub length, the share of supply points that are (n-1) secure (on a ring path or a
two-leg feeder), the normally-open links inside the districts and the normally-open ties to neighbouring
districts.

In [ ]:
names = groups["stubs and links"]
if len(names) == 1:
    print(f"No variant differs from {ref} in the stubs, links or ties; showing the reference.")
rows = []
for name in names:
    v = data[name]
    d = districts[districts["variant"] == name]
    ring_rows_v = v.rings[v.rings["kind"] == "ring"]
    links = v.lines[v.lines["role"] == "link"]
    rows.append({"variant": name, "circuit_km": d["circuit_km"].sum(), "cable_share": d["cable_km"].sum() / d["circuit_km"].sum(),
                 "overhead_km": d["overhead_km"].sum(), "n_stub_nodes": d["n_stub_nodes"].sum(),
                 "stub_km": v.rings["stub_km"].sum(),
                 "n1_secure_share": (d["n1_secure_share"] * d["n_supply_points"]).sum() / d["n_supply_points"].sum(),
                 "n_links": len(links), "link_km": links["length_km"].sum(),
                 "n_ties": d["n_ties"].sum(), "tie_km": d["tie_km"].sum(),
                 "ring_share_with_tie": ring_rows_v["has_tie"].mean() if len(ring_rows_v) else np.nan,
                 "km_per_station": d["circuit_km"].sum() / d["n_stations"].sum()})
display(mvu.pretty(pd.DataFrame(rows).set_index("variant")))

## Equipment and planning rules

Conductor types, planning utilization, nominal voltage and ring cap of the variants, with the electrical
extremes, repairs and band conflicts.

In [ ]:
names = groups["equipment and rules"]
if len(names) == 1:
    print(f"No variant differs from {ref} in the equipment or planning rules; showing the reference.")
rows = []
for name in names:
    v = data[name]
    d = districts[districts["variant"] == name]
    rows.append({"variant": name, "cable": v.snapshot("equipment.cable.name"),
                 "overhead_line": v.snapshot("equipment.overhead.name"),
                 "line_utilization": v.snapshot("loading.line_utilization"), "vn_kv": v.snapshot("voltage.vn_kv"),
                 "max_stations_per_ring": v.snapshot("topology.max_stations_per_ring"),
                 "max_loading_percent": d["max_loading_percent"].max(),
                 "max_loading_n1_percent": d["max_loading_n1_percent"].max(),
                 "max_drop_n1_percent": d["max_drop_n1_percent"].max(),
                 "n_repairs": d["n_repairs"].sum(), "n_without_violation": int((d["power_flow_status"] == "converged").sum()),
                 "n_band_conflicts": d["n_band_conflicts"].sum(), "n_tap_limit": d["n_tap_limit"].sum(),
                 "ring_members_mean": d["ring_members_mean"].mean()})
display(mvu.pretty(pd.DataFrame(rows).set_index("variant")))

## Maps side by side

The same district in the reference and the other variants (three per figure, coloured per ring; in the other
variants the district with the nearest substation).

In [ ]:
detail = grid_id if grid_id is not None else mvu.default_district(data[ref])
others = [n for n in variants if n != ref]
for i in range(0, max(len(others), 1), 2):
    chunk = [ref] + others[i:i + 2]
    fig = mvplot.plot_variants_side_by_side({n: data[n].layers for n in chunk}, detail,
                                            figsize=(9 * len(chunk), 9), basemap=basemap)

## Station set: two LV versions

The same MV variant on two LV versions, e.g. greenfield stations placed by the LV stage against brownfield
stations from OSM and LoD2 data (`USE_OPEN_TRANSFORMER_POSITIONS`). Set `compare_versions`.

In [ ]:
station_sets = {}
if not compare_versions or len(compare_versions) != 2:
    print("compare_versions is not set to two LV versions: the station-set comparison is skipped")
else:
    chosen = compare_variant or ref
    for version in compare_versions:
        _, found, listing_v = mvu.resolve(dbc, version, [chosen])
        if chosen not in found:
            continue
        parts_v = mvu.region_parts(dbc, listing_v.loc[listing_v["mv_variant_id"] == chosen, "plz_list"].iloc[0])
        station_sets[f"{version}/{chosen}"] = mvu.load_variant(dbc, version, chosen, parts=parts_v, regions=regions,
                                                                plz=plz, label=f"{version}/{chosen}")
    if len(station_sets) < 2:
        print(f"MV variant {chosen} does not exist in both LV versions {compare_versions}: comparison skipped")
if len(station_sets) == 2:
    set_districts = pd.concat([mvu.district_table(v) for v in station_sets.values()], ignore_index=True)
    set_rings = pd.concat([mvu.ring_table(v) for v in station_sets.values()], ignore_index=True)
    mvu.show_blocks(mvu.summary_table(set_districts, set_rings, by=["variant"]))
    sources_table = pd.concat({n: v.nodes.loc[v.nodes["kind"] == "station", "station_source"].value_counts()
                               for n, v in station_sets.items()}, axis=1).fillna(0).astype(int)
    display(Markdown("**Stations by source**"))
    display(sources_table)

In [ ]:
if len(station_sets) == 2:
    labels = list(station_sets)
    set_ring_rows = set_rings[set_rings["kind"] == "ring"].rename(columns={"n_members": "ring_members"})
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    mvplot.plot_metric_distribution(set_districts, "station_spacing_m", by="variant", order=labels, ax=axes[0],
                                    title="Station spacing per district (m)")
    mvplot.plot_metric_distribution(set_districts, "km_per_station", by="variant", order=labels, ax=axes[1],
                                    title="Circuit km per station per district")
    mvplot.plot_metric_distribution(set_ring_rows, "ring_members", by="variant", order=labels, ax=axes[2],
                                    title="Ring members per ring")
    fig.tight_layout()
    fig = mvplot.plot_tap_distribution({n: v.nodes for n, v in station_sets.items()}, figsize=(12, 5))
    first_set = station_sets[labels[0]]
    fig = mvplot.plot_variants_side_by_side({n: v.layers for n, v in station_sets.items()},
                                            mvu.default_district(first_set), figsize=(18, 9), basemap=basemap)

## Runtimes

Wall-clock time of the run (`mv_variant.summary`), summed district planning time and solver time.

In [ ]:
runtime = pd.DataFrame([{
    "variant": n, "elapsed_s": v.summary.get("elapsed_s"),
    "district_runtime_s": districts.loc[districts["variant"] == n, "runtime_s"].sum(),
    "solver_runtime_s": districts.loc[districts["variant"] == n, "solver_runtime_s"].sum(),
    "n_districts": len(v.grids), "n_stations": int(v.grids["n_stations"].sum())} for n, v in data.items()])
display(mvu.pretty(runtime.set_index("variant")))